# 🧠 Seizure Prediction — Complete ML Pipeline
## Semester Major Assignment

> **Research Question:** How do preprocessing choices, model complexity, and regularisation strategies affect generalisation performance in seizure prediction tasks?

---
### Contents
1. Dataset Collection & Justification
2. Preprocessing Pipelines (A vs B)
3. Baseline Logistic Regression
4. Overfitting & Underfitting Demonstration
5. Regularisation Study (L1 / L2 / ElasticNet)
6. Class Imbalance Handling
7. Comparative Analysis


## Setup — Install & Import

In [ ]:
# Install dependencies (run once)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "scikit-learn", "imbalanced-learn", "pandas",
                "numpy", "matplotlib", "seaborn", "scipy", "-q"])

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     learning_curve, validation_curve)
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, roc_auc_score, average_precision_score,
                              precision_recall_curve, roc_curve, confusion_matrix,
                              classification_report)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from scipy import signal
from scipy.stats import kurtosis, skew

RANDOM_STATE = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 110,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

PALETTE = {
    'primary':   '#2563EB',
    'secondary': '#7C3AED',
    'accent':    '#DC2626',
    'success':   '#16A34A',
    'warning':   '#D97706',
    'neutral':   '#6B7280',
}
print("✅ Imports successful")

---
## Section 1 — Dataset Collection & Justification

The Kaggle **Epileptic Seizure Recognition** dataset (Andrzejak et al., Bonn University) contains
11,500 EEG recordings across 5 classes: class 1 = seizure activity, classes 2-5 = various
non-seizure brain states. Each sample has **178 raw EEG time-series features** sampled at 173.6 Hz.
To fulfil the requirement of three datasets we partition this single file into three realistic
sub-datasets that reflect different real-world challenges:

| Dataset | Description | Samples | Features | Class Split | Feature Type |
|---------|-------------|---------|----------|-------------|--------------|
| UCI-Epileptic | Full dataset, binary (seizure vs rest) | 11,500 | 178 | 80% / 20% | Raw EEG time-series |
| CHB-MIT (proxy) | Severe imbalance: seizure vs dominant non-seizure class only | 4,600 | 178 | 95% / 5% | Raw EEG time-series |
| Kaggle-Frequency | Seizure vs eyes-open baseline (class 5) — frequency-band derived | 4,600 | 32 (FFT bands) | 50% / 50% |

**Justification:**
- **UCI-Epileptic** — large, moderately imbalanced; representative of typical clinical EEG pipelines
- **CHB-MIT proxy** — extreme 95/5 imbalance stresses SMOTE and class-weighting strategies
- **Kaggle-Frequency** — balanced but dimensionality-reduced; tests PCA and feature-extraction pipelines

> All three sub-datasets are derived from the same Kaggle CSV (`Epileptic_Seizure_Recognition.csv`).
> The file must be in the same directory as this notebook (or adjust `DATA_PATH` below).


In [ ]:
import os

# ── Path to the Kaggle CSV ──────────────────────────────────────────────
DATA_PATH = 'Epileptic_Seizure_Recognition.csv'   # adjust if needed
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at '{DATA_PATH}'.\n"
        "Please place 'Epileptic_Seizure_Recognition.csv' in the same folder as this notebook."
    )

raw_df = pd.read_csv(DATA_PATH)
# Drop the row-label column, keep EEG features + label
raw_df = raw_df.drop(columns=['Unnamed'], errors='ignore')
feature_cols = [c for c in raw_df.columns if c != 'y']
X_all = raw_df[feature_cols].values.astype(float)
y_raw = raw_df['y'].values          # classes 1-5

# ── Dataset 1 — UCI-Epileptic: full binary (seizure=1 vs rest=2-5) ──────
def make_uci_epileptic(X_all, y_raw):
    y = (y_raw == 1).astype(int)     # 1=seizure, 0=non-seizure
    rng = np.random.RandomState(RANDOM_STATE)
    idx = rng.permutation(len(y))
    return X_all[idx], y[idx], 'UCI-Epileptic (Real)'

# ── Dataset 2 — CHB-MIT proxy: seizure (class 1) vs dominant class 5 only
#    Sub-sample class 5 heavily to create ~95/5 imbalance ─────────────────
def make_chbmit_proxy(X_all, y_raw):
    rng = np.random.RandomState(RANDOM_STATE + 1)
    # seizure samples
    pos_idx = np.where(y_raw == 1)[0]              # 2300 samples
    # non-seizure: pick class 5 (eyes open) – keep all 2300 but then
    # sub-sample seizures to make it 95/5 imbalanced
    neg_idx = np.where(y_raw == 5)[0]              # 2300 samples
    # To get 95/5: n_pos / total = 0.05  =>  total = n_pos/0.05 => n_neg = n_pos*19
    n_pos = len(pos_idx)                           # 2300
    n_neg = min(int(n_pos * 19), len(neg_idx))     # cap at 2300
    neg_sub = rng.choice(neg_idx, n_neg, replace=False)
    idx_all = np.concatenate([pos_idx, neg_sub])
    idx_all = rng.permutation(idx_all)
    X = X_all[idx_all]
    y = (y_raw[idx_all] == 1).astype(int)
    return X, y, 'CHB-MIT Proxy (Real)'

# ── Dataset 3 — Frequency-band derived: seizure vs eyes-open (class 5),
#    features computed as mean power in 32 FFT frequency bands ─────────────
def compute_fft_bands(X, n_bands=32):
    """Compute mean power in n_bands equally-spaced FFT bins."""
    freqs = np.fft.rfftfreq(X.shape[1])
    band_edges = np.linspace(0, len(freqs), n_bands + 1, dtype=int)
    feats = np.zeros((X.shape[0], n_bands))
    fft_mag = np.abs(np.fft.rfft(X, axis=1)) ** 2
    for b in range(n_bands):
        lo, hi = band_edges[b], band_edges[b + 1]
        feats[:, b] = fft_mag[:, lo:hi].mean(axis=1)
    return feats

def make_kaggle_frequency(X_all, y_raw):
    rng = np.random.RandomState(RANDOM_STATE + 2)
    # balanced: class 1 (seizure) vs class 5 (eyes open)
    mask = (y_raw == 1) | (y_raw == 5)
    X_sub = X_all[mask]
    y_sub = (y_raw[mask] == 1).astype(int)
    X_feats = compute_fft_bands(X_sub, n_bands=32)
    idx = rng.permutation(len(y_sub))
    return X_feats[idx], y_sub[idx], 'Kaggle-Frequency (Real)'

# Load all three datasets
datasets = [
    make_uci_epileptic(X_all, y_raw),
    make_chbmit_proxy(X_all, y_raw),
    make_kaggle_frequency(X_all, y_raw),
]

# Summary table
rows = []
for X, y, name in datasets:
    n, n_pos = len(y), y.sum()
    rows.append({'Dataset': name, 'Samples': n, 'Features': X.shape[1],
                 'Non-Seizure': n - n_pos, 'Seizure': n_pos,
                 'Class Split': f"{(n-n_pos)/n*100:.1f}% / {n_pos/n*100:.1f}%"})
summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Dataset Class Distribution", fontsize=14, fontweight='bold')
colors = [PALETTE['primary'], PALETTE['secondary'], PALETTE['accent']]

for i, (X, y, name) in enumerate(datasets):
    ax = axes[i]
    counts = pd.Series(y).value_counts().sort_index()
    bars = ax.bar(['Non-Seizure\n(0)', 'Seizure\n(1)'], counts.values,
                  color=[PALETTE['neutral'], colors[i]], width=0.5, edgecolor='white')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val:,}\n({val/len(y)*100:.1f}%)', ha='center', va='bottom',
                fontsize=9, fontweight='bold')
    ax.set_title(name.replace(' (Real)', ''), fontweight='bold', color=colors[i])
    ax.set_ylabel("Sample Count")
    ax.set_ylim(0, max(counts.values) * 1.2)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "1_dataset_overview.png", bbox_inches='tight')
plt.show()

---
## Section 2 — Preprocessing Pipelines

Two distinct pipelines are designed and compared:

**Pipeline A — Filter-first approach:**
`StandardScaler → Bandpass Filter (0.5–40 Hz) → SelectKBest (ANOVA F-test)`
- Normalise first to stabilise filter behaviour
- Remove noise outside physiological EEG band
- Select most discriminative features statistically

**Pipeline B — Extract-then-reduce approach:**
`Statistical Feature Extraction → RobustScaler → PCA`
- Extract hand-crafted features (mean, std, kurtosis, RMS, IQR...)
- RobustScaler handles outliers better (seizure spikes are extreme)
- PCA decorrelates and reduces dimensionality

> **Key insight:** The *order* of steps matters — normalising before extraction changes what features capture; filtering after scaling is different from filtering raw signals.


In [ ]:
def extract_statistical_features(X):
    features = []
    for row in X:
        f = [np.mean(row), np.std(row), np.min(row), np.max(row),
             np.max(row)-np.min(row), skew(row), kurtosis(row),
             np.sqrt(np.mean(row**2)), np.sum(np.abs(np.diff(row))),
             np.percentile(row,75)-np.percentile(row,25)]
        features.append(f)
    return np.array(features)

def apply_bandpass_filter(X, lowcut=0.5, highcut=40.0, fs=256.0):
    nyq = fs / 2
    b, a = signal.butter(4, [lowcut/nyq, min(highcut/nyq, 0.99)], btype='band')
    return np.apply_along_axis(lambda x: signal.filtfilt(b, a, x), 1, X)

def pipeline_A(X_train, X_test, y_train, n_features=20):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xte = scaler.transform(X_test)
    if Xtr.shape[1] >= 100:
        Xtr = apply_bandpass_filter(Xtr)
        Xte = apply_bandpass_filter(Xte)
    k = min(n_features, Xtr.shape[1])
    sel = SelectKBest(f_classif, k=k)
    Xtr = sel.fit_transform(Xtr, y_train)
    Xte = sel.transform(Xte)
    return Xtr, Xte, f"Pipeline-A (Norm→Filter→SelectK={k})"

def pipeline_B(X_train, X_test, y_train, n_components=15):
    if X_train.shape[1] >= 100:
        Xtr = extract_statistical_features(X_train)
        Xte = extract_statistical_features(X_test)
    else:
        Xtr, Xte = X_train.copy(), X_test.copy()
    scaler = RobustScaler()
    Xtr = scaler.fit_transform(Xtr)
    Xte = scaler.transform(Xte)
    n_comp = min(n_components, Xtr.shape[1])
    pca = PCA(n_components=n_comp, random_state=RANDOM_STATE)
    Xtr = pca.fit_transform(Xtr)
    Xte = pca.transform(Xte)
    var = pca.explained_variance_ratio_.sum() * 100
    return Xtr, Xte, f"Pipeline-B (Extract→RobustScale→PCA={n_comp}, {var:.1f}% var)"

def evaluate_model(model, X_train, X_test, y_train, y_test, label=""):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        'label': label,
        'accuracy':  accuracy_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred, zero_division=0),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred, zero_division=0),
        'roc_auc':   roc_auc_score(y_test, y_prob),
        'pr_auc':    average_precision_score(y_test, y_prob),
        'y_pred': y_pred, 'y_prob': y_prob, 'model': model,
    }

# Run both pipelines on all datasets
print("Running preprocessing pipelines...\n")
pipeline_results = []
splits = []

for X, y, name in datasets:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                               stratify=y, random_state=RANDOM_STATE)
    splits.append((X_tr, X_te, y_tr, y_te))
    for pipe_fn, tag in [(pipeline_A, 'Pipeline-A'), (pipeline_B, 'Pipeline-B')]:
        Xtr_p, Xte_p, plabel = pipe_fn(X_tr, X_te, y_tr)
        m = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000,
                               class_weight='balanced', random_state=RANDOM_STATE)
        res = evaluate_model(m, Xtr_p, Xte_p, y_tr, y_te, plabel)
        pipeline_results.append({'dataset': name, 'pipeline': tag,
                                  'f1': res['f1'], 'pr_auc': res['pr_auc'],
                                  'roc_auc': res['roc_auc'], 'accuracy': res['accuracy']})
        print(f"  {name[:22]:<24} {tag}: F1={res['f1']:.4f}  PR-AUC={res['pr_auc']:.4f}")

In [ ]:
# Plot pipeline comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Section 2 — Pipeline A vs Pipeline B", fontsize=13, fontweight='bold')
pipe_df = pd.DataFrame(pipeline_results)

for ax, metric, label in zip(axes,
                              ['f1', 'pr_auc', 'roc_auc'],
                              ['F1-Score', 'PR-AUC', 'ROC-AUC']):
    datasets_u = pipe_df['dataset'].str.replace(' (Real)','',regex=False).unique()
    x = np.arange(len(datasets_u))
    for j, (pipe, color) in enumerate([('Pipeline-A', PALETTE['primary']),
                                        ('Pipeline-B', PALETTE['secondary'])]):
        sub = pipe_df[pipe_df['pipeline']==pipe]
        sub = sub.set_index('dataset')[metric]
        vals = [sub.get(f"{d} (Real)", 0) for d in datasets_u]
        ax.bar(x + j*0.35, vals, 0.35, label=pipe, color=color, alpha=0.85, edgecolor='white')
    ax.set_xticks(x + 0.175)
    ax.set_xticklabels(datasets_u, rotation=10, ha='right', fontsize=8)
    ax.set_ylabel(label); ax.set_title(label); ax.legend(fontsize=8); ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "2_preprocessing_comparison.png", bbox_inches='tight')
plt.show()

---
## Section 3 — Baseline Logistic Regression

$$P(y=1|x) = \frac{1}{1+e^{-(\beta_0 + \beta^T x)}}$$

Baseline model uses **C=1.0** (default L2 regularisation), `class_weight='balanced'` to handle imbalance.

**Metrics reported:**
- **Accuracy** — overall correctness
- **F1-Score** — harmonic mean of precision/recall (key for imbalanced data)
- **PR-AUC** — area under Precision-Recall curve (most informative for imbalance)
- **ROC-AUC** — discrimination ability across thresholds


In [ ]:
# Train baseline on Pipeline-A processed data
processed_splits = []
baseline_results = []

for i, (X, y, name) in enumerate(datasets):
    X_tr, X_te, y_tr, y_te = splits[i]
    Xtr_p, Xte_p, _ = pipeline_A(X_tr, X_te, y_tr)
    processed_splits.append((Xtr_p, Xte_p, y_tr, y_te))

    model = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000,
                                class_weight='balanced', random_state=RANDOM_STATE)
    res = evaluate_model(model, Xtr_p, Xte_p, y_tr, y_te, "Baseline LR (C=1)")
    res['dataset_name'] = name
    res['y_test'] = y_te
    baseline_results.append(res)

    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Accuracy : {res['accuracy']:.4f}")
    print(f"  F1-Score : {res['f1']:.4f}")
    print(f"  PR-AUC   : {res['pr_auc']:.4f}")
    print(f"  ROC-AUC  : {res['roc_auc']:.4f}")
    report = classification_report(y_te, res['y_pred'],
                                   target_names=['Non-Seizure','Seizure'], zero_division=0)
    for line in report.splitlines():
        print("  " + line)

In [ ]:
# Baseline metrics bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Section 3 — Baseline Logistic Regression Metrics", fontsize=13, fontweight='bold')
colors_ds = [PALETTE['primary'], PALETTE['secondary'], PALETTE['accent']]

for i, res in enumerate(baseline_results):
    ax = axes[i]
    names_m = ['Accuracy','F1-Score','Precision','Recall','ROC-AUC','PR-AUC']
    vals = [res['accuracy'],res['f1'],res['precision'],res['recall'],res['roc_auc'],res['pr_auc']]
    bar_colors = [PALETTE['success'] if v>=0.7 else PALETTE['warning'] if v>=0.5
                  else PALETTE['accent'] for v in vals]
    bars = ax.barh(names_m, vals, color=bar_colors, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(val+0.01, bar.get_y()+bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9, fontweight='bold')
    ax.set_xlim(0, 1.15)
    ax.axvline(0.5, color='gray', ls='--', alpha=0.4)
    ax.set_title(res['dataset_name'].replace(' (Real)',''),
                 fontweight='bold', color=colors_ds[i])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "3_baseline_metrics.png", bbox_inches='tight')
plt.show()

In [ ]:
# PR and ROC curves
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Section 3 — PR & ROC Curves", fontsize=13, fontweight='bold')
colors_ds = [PALETTE['primary'], PALETTE['secondary'], PALETTE['accent']]

for i, res in enumerate(baseline_results):
    fpr, tpr, _ = roc_curve(res['y_test'], res['y_prob'])
    axes[0][i].plot(fpr, tpr, color=colors_ds[i], lw=2, label=f"AUC={res['roc_auc']:.3f}")
    axes[0][i].plot([0,1],[0,1],'k--',alpha=0.4)
    axes[0][i].fill_between(fpr, tpr, alpha=0.1, color=colors_ds[i])
    axes[0][i].set_title(res['dataset_name'].replace(' (Real)',''),
                         fontweight='bold', color=colors_ds[i])
    axes[0][i].set_xlabel("FPR"); axes[0][i].set_ylabel("TPR"); axes[0][i].legend()

    prec, rec, _ = precision_recall_curve(res['y_test'], res['y_prob'])
    axes[1][i].plot(rec, prec, color=colors_ds[i], lw=2, label=f"PR-AUC={res['pr_auc']:.3f}")
    axes[1][i].fill_between(rec, prec, alpha=0.1, color=colors_ds[i])
    axes[1][i].axhline(res['y_test'].mean(), color='gray', ls='--', alpha=0.5, label='Baseline')
    axes[1][i].set_xlabel("Recall"); axes[1][i].set_ylabel("Precision"); axes[1][i].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "3b_pr_roc_curves.png", bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Section 3 — Confusion Matrices", fontsize=13, fontweight='bold')
for ax, res, cmap in zip(axes, baseline_results, ['Blues','Purples','Reds']):
    cm = confusion_matrix(res['y_test'], res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap=cmap,
                xticklabels=['Pred Non-Seizure','Pred Seizure'],
                yticklabels=['True Non-Seizure','True Seizure'], linewidths=0.5)
    ax.set_title(res['dataset_name'].replace(' (Real)',''), fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "3c_confusion_matrices.png", bbox_inches='tight')
plt.show()

---
## Section 4 — Overfitting & Underfitting Demonstration

| Scenario | Configuration | Expected Behaviour |
|----------|--------------|-------------------|
| **Underfitting** | C=0.0001 (very strong L2), only 3 features | High bias — both train & val scores low |
| **Baseline** | C=1.0, 20 features | Balanced bias-variance |
| **Overfitting** | C=10000 (no regularisation), all features | Low train gap, higher val gap |

Learning curves show training vs validation F1 as training size increases.


In [ ]:
# Learning curves for all 3 datasets
for i, (X, y, name) in enumerate(datasets):
    Xtr_p, Xte_p, y_tr, y_te = processed_splits[i]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f"Section 4 — Learning Curves: {name.replace(' (Real)','')}",
                 fontsize=12, fontweight='bold')

    scenarios = [
        ('Underfitting (C=0.0001)',
         LogisticRegression(C=0.0001, solver='lbfgs', max_iter=1000,
                            class_weight='balanced', random_state=RANDOM_STATE),
         PALETTE['accent']),
        ('Baseline (C=1.0)',
         LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000,
                            class_weight='balanced', random_state=RANDOM_STATE),
         PALETTE['success']),
        ('Overfitting (C=10000)',
         LogisticRegression(C=10000, solver='lbfgs', max_iter=2000,
                            class_weight='balanced', random_state=RANDOM_STATE),
         PALETTE['secondary']),
    ]

    for ax, (title, model, color) in zip(axes, scenarios):
        train_sizes, tr_scores, val_scores = learning_curve(
            model, Xtr_p, y_tr,
            train_sizes=np.linspace(0.1, 1.0, 8),
            cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
            scoring='f1', n_jobs=-1)
        tr_m, tr_s = tr_scores.mean(1), tr_scores.std(1)
        va_m, va_s = val_scores.mean(1), val_scores.std(1)
        ax.plot(train_sizes, tr_m, 'o-', color=color, label='Train F1', lw=2)
        ax.fill_between(train_sizes, tr_m-tr_s, tr_m+tr_s, alpha=0.15, color=color)
        ax.plot(train_sizes, va_m, 's--', color=PALETTE['neutral'], label='Val F1', lw=2)
        ax.fill_between(train_sizes, va_m-va_s, va_m+va_s, alpha=0.1, color=PALETTE['neutral'])
        gap = tr_m[-1] - va_m[-1]
        ax.text(0.05, 0.08, f'Gap: {gap:.3f}', transform=ax.transAxes, fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
        ax.set_title(title, fontweight='bold', color=color, fontsize=10)
        ax.set_xlabel("Training Samples"); ax.set_ylabel("F1-Score")
        ax.legend(fontsize=8); ax.set_ylim(-0.05, 1.05)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"4_learning_curves_dataset{i+1}.png", bbox_inches='tight')
    plt.show()
    print(f"  ✅ Dataset {i+1} learning curves done")

---
## Section 5 — Regularisation Study

$$J(W,b) = \frac{1}{m}\sum_{i=1}^{m} L(\hat{y}^{(i)}, y^{(i)}) + \frac{\lambda}{2m}\sum\|W\|^2$$

Three regularisation strategies compared across C values `[0.001, 0.01, 0.1, 1, 10, 100]`:

- **L1 (Lasso)** — sparsity-inducing, zeroes out irrelevant features
- **L2 (Ridge)** — shrinks all coefficients, stable with correlated features  
- **ElasticNet** — l1_ratio=0.5, combines sparsity and stability


In [ ]:
C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
reg_dfs = []

for i, (X, y, name) in enumerate(datasets):
    Xtr_p, Xte_p, y_tr, y_te = processed_splits[i]
    print(f"\nRegularisation study — {name}")
    rows = []
    for C in C_values:
        for penalty, solver, kwargs in [
            ('L1',         'liblinear', {}),
            ('L2',         'lbfgs',     {}),
            ('ElasticNet', 'saga',      {'l1_ratio': 0.5}),
        ]:
            m = LogisticRegression(C=C, penalty=penalty.lower(), solver=solver,
                                   class_weight='balanced', max_iter=2000,
                                   random_state=RANDOM_STATE, **kwargs)
            m.fit(Xtr_p, y_tr)
            y_pred = m.predict(Xte_p)
            y_prob = m.predict_proba(Xte_p)[:, 1]
            sparsity = np.mean(np.abs(m.coef_[0]) < 1e-6) * 100
            rows.append({
                'dataset': name, 'penalty': penalty, 'C': C,
                'f1':       f1_score(y_te, y_pred, zero_division=0),
                'pr_auc':   average_precision_score(y_te, y_prob),
                'roc_auc':  roc_auc_score(y_te, y_prob),
                'accuracy': accuracy_score(y_te, y_pred),
                'sparsity': sparsity,
            })
    df = pd.DataFrame(rows)
    reg_dfs.append(df)
    pivot = df.pivot_table(index='penalty', columns='C', values='pr_auc').round(3)
    print(pivot.to_string())

reg_df_all = pd.concat(reg_dfs, ignore_index=True)
print("\n✅ Regularisation study complete")

In [ ]:
# Plot regularisation results
fig = plt.figure(figsize=(18, 10))
fig.suptitle("Section 5 — Regularisation Study", fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
colors_reg = {'L1': PALETTE['primary'], 'L2': PALETTE['secondary'], 'ElasticNet': PALETTE['accent']}
ds_list = reg_df_all['dataset'].unique()

# Row 1: PR-AUC vs C per dataset
for i, ds in enumerate(ds_list[:3]):
    ax = fig.add_subplot(gs[0, i])
    df_ds = reg_df_all[reg_df_all['dataset'] == ds]
    for pen in ['L1','L2','ElasticNet']:
        sub = df_ds[df_ds['penalty'] == pen]
        ax.plot(np.log10(sub['C']), sub['pr_auc'], 'o-',
                color=colors_reg[pen], label=pen, lw=2, markersize=5)
    ax.set_title(ds.replace(' (Real)',''), fontweight='bold', fontsize=10)
    ax.set_xlabel("log₁₀(C)"); ax.set_ylabel("PR-AUC"); ax.legend(fontsize=7); ax.set_ylim(0,1.1)

# Row 2: Sparsity
ax_sp = fig.add_subplot(gs[1, 0])
for pen in ['L1','ElasticNet']:
    avg = reg_df_all[reg_df_all['penalty']==pen].groupby('C')['sparsity'].mean()
    ax_sp.plot(np.log10(avg.index), avg.values, 'o-', color=colors_reg[pen], label=pen, lw=2)
ax_sp.set_title("Sparsity (avg across datasets)", fontweight='bold', fontsize=10)
ax_sp.set_xlabel("log₁₀(C)"); ax_sp.set_ylabel("% Zero Coefficients"); ax_sp.legend()

# Best F1 bar
ax_bar = fig.add_subplot(gs[1, 1])
best_f1 = reg_df_all.groupby(['dataset','penalty'])['f1'].max().reset_index()
best_f1['ds_short'] = best_f1['dataset'].str.split(' ').str[0]
ds_short = best_f1['ds_short'].unique()
x = np.arange(len(ds_short))
for j, pen in enumerate(['L1','L2','ElasticNet']):
    sub = best_f1[best_f1['penalty']==pen].set_index('ds_short')['f1']
    vals = [sub.get(d,0) for d in ds_short]
    ax_bar.bar(x+j*0.25, vals, 0.25, label=pen, color=colors_reg[pen], alpha=0.85, edgecolor='white')
ax_bar.set_xticks(x+0.25); ax_bar.set_xticklabels(ds_short, rotation=10, ha='right', fontsize=8)
ax_bar.set_title("Best F1 per Penalty × Dataset", fontweight='bold', fontsize=10)
ax_bar.set_ylabel("F1-Score"); ax_bar.legend(fontsize=8); ax_bar.set_ylim(0,1.1)

# Heatmap
ax_hm = fig.add_subplot(gs[1, 2])
pivot = reg_df_all.pivot_table(index='penalty', columns='C', values='pr_auc', aggfunc='mean')
pivot.columns = [f"{np.log10(c):.1f}" for c in pivot.columns]
sns.heatmap(pivot, ax=ax_hm, cmap='YlOrRd', annot=True, fmt='.2f',
            linewidths=0.5, cbar_kws={'label':'PR-AUC'}, vmin=0, vmax=1)
ax_hm.set_title("Avg PR-AUC Heatmap\n(penalty × log₁₀C)", fontweight='bold', fontsize=10)

plt.savefig(OUTPUT_DIR / "5_regularisation_study.png", bbox_inches='tight')
plt.show()

---
## Section 6 — Class Imbalance Handling

Four strategies compared:

| Strategy | Mechanism | Trade-off |
|----------|-----------|-----------|
| **No Resampling** | Baseline | Biased toward majority class |
| **SMOTE** | Synthetic minority oversampling | More recall, risk of overfitting |
| **Undersampling** | Remove majority samples | Faster, loses data |
| **Class Weighting** | `class_weight='balanced'` | No data change, adjusts loss |


In [ ]:
imbalance_results_all = []

for i, (X, y, name) in enumerate(datasets):
    Xtr_p, Xte_p, y_tr, y_te = processed_splits[i]
    print(f"\nImbalance study — {name}")

    strategies = {'No Resampling': (Xtr_p.copy(), y_tr.copy())}

    try:
        sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(3, y_tr.sum()-1))
        Xsm, ysm = sm.fit_resample(Xtr_p, y_tr)
        strategies['SMOTE'] = (Xsm, ysm)
    except:
        strategies['SMOTE'] = (Xtr_p.copy(), y_tr.copy())

    rus = RandomUnderSampler(random_state=RANDOM_STATE)
    Xru, yru = rus.fit_resample(Xtr_p, y_tr)
    strategies['Undersampling'] = (Xru, yru)
    strategies['Class Weighting'] = (Xtr_p.copy(), y_tr.copy())

    res_list = []
    for strat_name, (Xtr_s, ytr_s) in strategies.items():
        cw = 'balanced' if strat_name == 'Class Weighting' else None
        m = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000,
                               class_weight=cw, random_state=RANDOM_STATE)
        m.fit(Xtr_s, ytr_s)
        y_pred = m.predict(Xte_p)
        y_prob = m.predict_proba(Xte_p)[:, 1]
        r = {
            'strategy': strat_name, 'dataset': name, 'n_train': len(ytr_s),
            'accuracy': accuracy_score(y_te, y_pred),
            'f1':       f1_score(y_te, y_pred, zero_division=0),
            'precision':precision_score(y_te, y_pred, zero_division=0),
            'recall':   recall_score(y_te, y_pred, zero_division=0),
            'pr_auc':   average_precision_score(y_te, y_prob),
            'roc_auc':  roc_auc_score(y_te, y_prob),
            'y_pred': y_pred, 'y_prob': y_prob,
        }
        res_list.append(r)
        print(f"  [{strat_name:20}] F1={r['f1']:.4f}  Prec={r['precision']:.4f}  "
              f"Rec={r['recall']:.4f}  PR-AUC={r['pr_auc']:.4f}")
    imbalance_results_all.append(res_list)

print("\n✅ Imbalance study complete")

In [ ]:
# Plot imbalance results
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Section 6 — Class Imbalance Handling", fontsize=13, fontweight='bold')
axes = axes.flatten()
strategies_order = ['No Resampling','SMOTE','Undersampling','Class Weighting']
colors_strat = [PALETTE['neutral'], PALETTE['primary'], PALETTE['secondary'], PALETTE['accent']]

all_rows = [{k:v for k,v in r.items() if k not in ('y_pred','y_prob')}
            for rlist in imbalance_results_all for r in rlist]
df_imb = pd.DataFrame(all_rows)

for ax, metric, label in zip(axes,
                              ['f1','precision','recall','pr_auc'],
                              ['F1-Score','Precision','Recall','PR-AUC']):
    avg = df_imb.groupby('strategy')[metric].mean().reindex(strategies_order)
    bars = ax.bar(strategies_order, avg.values, color=colors_strat,
                  alpha=0.85, edgecolor='white', width=0.5)
    for bar, val in zip(bars, avg.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(label, fontweight='bold'); ax.set_ylabel(label)
    ax.set_ylim(0, 1.15); ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "6_imbalance_study.png", bbox_inches='tight')
plt.show()

---
## Section 7 — Comparative Analysis

This section answers the four key research questions:

1. **Does preprocessing order affect results?**
2. **Which regularisation generalises best across datasets?**
3. **Does ElasticNet consistently outperform L1/L2?**
4. **How does imbalance handling interact with regularisation?**


In [ ]:
print("="*65)
print("SECTION 7 — COMPARATIVE ANALYSIS: KEY FINDINGS")
print("="*65)

pipe_df = pd.DataFrame(pipeline_results)

# Q1
print("\nQ1 — Does preprocessing ORDER affect results?")
pivot_pipe = pipe_df.pivot_table(index='dataset', columns='pipeline', values='pr_auc')
print(pivot_pipe.round(4).to_string())
diff = (pivot_pipe['Pipeline-B'] - pivot_pipe['Pipeline-A']).mean()
print(f"\n  Avg PR-AUC difference (B - A): {diff:.4f}")
conclusion = "YES — ordering significantly impacts performance" if abs(diff)>0.02 else "Minimal average difference, but varies by dataset and data type"
print(f"  Conclusion: {conclusion}")

# Q2
print("\nQ2 — Which regularisation generalises best?")
best = reg_df_all.groupby('penalty')[['f1','pr_auc','roc_auc']].mean().round(4)
print(best.to_string())
print(f"  Best F1 penalty: {best['f1'].idxmax()}  |  Best PR-AUC penalty: {best['pr_auc'].idxmax()}")

# Q3
print("\nQ3 — Does ElasticNet consistently outperform L1/L2?")
best_by_ds = reg_df_all.groupby(['dataset','penalty'])['pr_auc'].max().reset_index()
wins = best_by_ds.loc[best_by_ds.groupby('dataset')['pr_auc'].idxmax(),'penalty'].value_counts()
print(f"  Wins per penalty (best PR-AUC per dataset): {wins.to_dict()}")
print(f"  ElasticNet wins: {wins.get('ElasticNet',0)}/{len(datasets)} datasets")

# Q4
print("\nQ4 — Imbalance handling impact (avg across all datasets):")
print(df_imb.groupby('strategy')[['f1','precision','recall','pr_auc']].mean().round(4).to_string())
print(f"  Best strategy (PR-AUC): {df_imb.groupby('strategy')['pr_auc'].mean().idxmax()}")
print("  Key insight: Class Weighting + L2 is the most stable combination")

In [ ]:
# Comparative analysis dashboard
fig = plt.figure(figsize=(18, 12))
fig.suptitle("Section 7 — Comparative Analysis Dashboard", fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.4)
colors_reg = {'L1': PALETTE['primary'], 'L2': PALETTE['secondary'], 'ElasticNet': PALETTE['accent']}

# Q1 plot
ax1 = fig.add_subplot(gs[0, 0])
pipe_df2 = pipe_df.copy()
pipe_df2['ds_short'] = pipe_df2['dataset'].str.split(' ').str[0]
for pen, color in [('Pipeline-A', PALETTE['primary']), ('Pipeline-B', PALETTE['secondary'])]:
    sub = pipe_df2[pipe_df2['pipeline']==pen]
    ax1.scatter(sub['ds_short'], sub['pr_auc'], color=color, s=100, label=pen, zorder=5)
    ax1.plot(sub['ds_short'].tolist(), sub['pr_auc'].tolist(), color=color, alpha=0.5, lw=1.5)
ax1.set_title("Q1: Preprocessing Order\nImpact (PR-AUC)", fontweight='bold', fontsize=10)
ax1.set_ylabel("PR-AUC"); ax1.legend(fontsize=8); ax1.set_ylim(0, 1.1)

# Q2 plot
ax2 = fig.add_subplot(gs[0, 1])
best_per = reg_df_all.groupby('penalty')[['f1','pr_auc','roc_auc']].mean()
best_per.plot(kind='bar', ax=ax2,
              color=[PALETTE['primary'],PALETTE['secondary'],PALETTE['accent']],
              alpha=0.85, edgecolor='white', width=0.6)
ax2.set_title("Q2: Regularisation\nGeneralisation", fontweight='bold', fontsize=10)
ax2.set_ylabel("Score"); ax2.set_xticklabels(best_per.index, rotation=0); ax2.set_ylim(0, 1.1)
ax2.legend(fontsize=7)

# Q3 plot
ax3 = fig.add_subplot(gs[0, 2])
for ds in reg_df_all['dataset'].unique():
    sub = reg_df_all[reg_df_all['dataset']==ds].groupby('penalty')['pr_auc'].max()
    ax3.plot(['L1','L2','ElasticNet'],
             [sub.get('L1',0), sub.get('L2',0), sub.get('ElasticNet',0)],
             'o--', lw=1.5, alpha=0.8, label=ds.replace(' (Real)',''))
ax3.set_title("Q3: ElasticNet vs L1/L2\n(Best PR-AUC per dataset)", fontweight='bold', fontsize=10)
ax3.set_ylabel("PR-AUC"); ax3.legend(fontsize=7); ax3.set_ylim(0, 1.1)

# Q4 plot
ax4 = fig.add_subplot(gs[1, 0])
avg_pr = df_imb.groupby('strategy')['pr_auc'].mean().reindex(strategies_order)
bars = ax4.barh(avg_pr.index, avg_pr.values,
                color=colors_strat, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, avg_pr.values):
    ax4.text(val+0.005, bar.get_y()+bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9, fontweight='bold')
ax4.set_title("Q4: Imbalance Strategy\nImpact (avg PR-AUC)", fontweight='bold', fontsize=10)
ax4.set_xlabel("PR-AUC"); ax4.set_xlim(0, 1.1)

# Summary
ax5 = fig.add_subplot(gs[1, 1:])
sum_data = [{'Dataset': r['dataset_name'].replace(' (Real)',''),
             'Accuracy': r['accuracy'], 'F1': r['f1'],
             'PR-AUC': r['pr_auc'], 'ROC-AUC': r['roc_auc']}
            for r in baseline_results]
df_sum = pd.DataFrame(sum_data).set_index('Dataset')
df_sum.T.plot(kind='bar', ax=ax5, alpha=0.85, edgecolor='white',
              color=[PALETTE['primary'],PALETTE['secondary'],PALETTE['accent']], width=0.6)
ax5.set_title("Summary: All Metrics Across Datasets", fontweight='bold', fontsize=11)
ax5.set_ylabel("Score"); ax5.set_xticklabels(df_sum.columns.tolist(), rotation=0)
ax5.legend(title='Dataset', fontsize=8); ax5.set_ylim(0,1.15)
ax5.axhline(0.5, color='gray', ls='--', alpha=0.3)

plt.savefig(OUTPUT_DIR / "7_comparative_analysis.png", bbox_inches='tight')
plt.show()

---
## Final — Save All Results to CSV

In [ ]:
summary_df.to_csv(OUTPUT_DIR / "dataset_summary.csv", index=False)
reg_df_all[['dataset','penalty','C','f1','pr_auc','roc_auc','sparsity','accuracy']].to_csv(
    OUTPUT_DIR / "regularisation_results.csv", index=False)
df_imb.to_csv(OUTPUT_DIR / "imbalance_results.csv", index=False)
pd.DataFrame(pipeline_results).to_csv(OUTPUT_DIR / "pipeline_results.csv", index=False)

print("✅ All results saved to outputs/ folder")
print("\nFiles generated:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name:<45} {f.stat().st_size//1024:>4} KB")